# Taller de Optimización Lineal: Despacho Eléctrico y Portafolio de Crédito

Este notebook contiene la formulación matemática y la implementación en Python (usando SciPy y PuLP) para resolver y analizar dos problemas de programación lineal, incluyendo su análisis de dualidad y sensibilidad detallado.

## Registro de cambios respecto a la v1 (`Taller3optimizacionV1.ipynb`)

Esta v2 mantiene TODO el código original de la v1 que estaba correcto (primal y dual de
ambos problemas con SciPy y PuLP) y agrega lo que faltaba para cumplir el enunciado del taller.
No se borró ningún cálculo que ya funcionaba; solo se reemplazaron dos secciones mal
enfocadas (ver "CAMBIADO" abajo).

**MANTENIDO SIN CAMBIOS** (celdas originales, siguen igual):
- Formulación primal/dual en markdown de ambos problemas.
- Resolución primal (SciPy + PuLP) y dual explícito (SciPy) de ambos problemas.
- Verificación de dualidad fuerte del Problema 1.
- Verificación de residuos primales sin solver (numpy).

**AGREGADO** (no existía nada de esto en la v1):
- Conversión explícita de unidades (miles USD ↔ millones USD, miles USD/tCO2 ↔ USD/tCO2).
- Pregunta 4 completa: comparación fila por fila entre `ineqlin.marginals` del solver y el
  dual explícito, con nota sobre el cambio de signo al convertir `≥` en `≤` (y, en Problema 2,
  al convertir `max` en `min`).
- Pregunta 5 completa: costos reducidos (`lower.marginals` / `upper.marginals`) para las
  5 variables de cada problema.
- V.1: tabla de trazabilidad enunciado↔código y verificación dimensional numérica (faltaba
  por completo).
- V.4 para el Problema 2 (antes solo existía para el Problema 1).
- V.5 completa: holgura complementaria (restricciones y variables) y factibilidad dual,
  consolidada para ambos problemas (no existía nada de esto).
- V.7: 3 pruebas de coherencia — monotonía, caso extremo y verificación de unicidad
  (no existía ninguna).
- Pregunta 9 completa (los 3 literales) de ambos problemas — estaba en 0% en la v1.
- V.8 reescrita como auditoría real: incluye un error concreto y verificable que se encontró
  al comparar el dual explícito contra `ineqlin.marginals` (un error de signo, el error
  frecuente #5 del anexo del taller), en vez de una nota genérica de limitaciones.

**CAMBIADO / REEMPLAZADO** (existía en la v1 pero estaba mal enfocado o mal etiquetado):
- El título "### V.1 Verificación de Factibilidad Primal..." se renombró a **V.2** (ese es
  su contenido real según el PDF del taller; V.1 es la trazabilidad, que antes no existía).
- El título "### V.5 Análisis de Sensibilidad por Perturbación..." se renombró a **V.6**
  (V.5 real es holgura complementaria, agregada aparte).
- El código de perturbación finita (antes solo tabulaba `Z(b+δ)` en un rango arbitrario,
  sin encontrar el punto de quiebre) se reemplazó por una función que detecta
  automáticamente dónde deja de cumplirse `Z(b+δ)-Z(b) = y·δ`, que es lo que pide V.6
  explícitamente. Se aplica a 2 recursos por problema (mínimo exigido) más una tabla
  resumen para el resto.

**Nota de ejecución:** no pude correr `Restart & Run All` de este notebook en este entorno
porque no tengo acceso a internet para instalar PuLP aquí; sí verifiqué por fuera, con
scipy solo, que todos los valores numéricos usados en las celdas nuevas (duales, costos
reducidos, puntos de quiebre, escenarios de la Pregunta 9) son correctos. Antes de entregar,
corre tú el notebook completo de principio a fin una vez para confirmar que PuLP se instala
sin problemas en tu entorno.

## Correcciones v3 (sobre el notebook ya ejecutado que me compartiste)

Al revisar tu v2 YA EJECUTADA encontré un **bug real en mi propio código** (no tuyo):

- **Celda de V.5 (holgura complementaria de variables):** el chequeo usaba `x_j * costo_reducido_j = 0`
  para TODAS las variables, pero esa fórmula solo es válida cuando `x_j` está en su cota INFERIOR (0).
  Para G1, G2 (Problema 1) y L1, L2, L4 (Problema 2), que están en su cota SUPERIOR, la condición correcta
  es `(cota - x_j) * costo_reducido_j = 0`. Por eso tu v2 ejecutada mostraba "NO CUMPLE" en ambos problemas
  — era un falso negativo mío, no un error del modelo. **Corregido.**

También apliqué dos mejoras que había recomendado en la revisión anterior pero que no llegué a meter en
el archivo:
- **V.2 (residuos primales):** ahora la tolerancia `tol` se usa explícitamente para verificar TODAS las
  restricciones (antes solo la igualdad) y se agregó la verificación de no negatividad y de cotas superiores.
- **Formulación P1:** se corrigió la notación contradictoria entre `μ_j⁺≥0` (en la lista de cotas) y
  `w_j≤0` (en la sección dual) — ambas se referían a la misma variable con signo distinto sin explicarlo.



## PROBLEMA 1: Despacho Eléctrico

### 1. Formulación Primal (Matemática)
Sea $x_j$ la generación de la planta $j$ en GWh/semana para $j \in \{1, 2, 3, 4, 5\}$ (Hidro, Eólica, Gas, Carbón, Diésel).

**Función Objetivo (Minimizar Costo Variable Total en miles USD/semana):**
$$\min z = 12x_1 + 8x_2 + 45x_3 + 32x_4 + 95x_5$$

**Restricciones:**
1. **Demanda:**
   $$x_1 + x_2 + x_3 + x_4 + x_5 = 780 \quad (\text{GWh}) \quad [y_1 \in \mathbb{R}]$$
2. **Emisiones:**
   $$0x_1 + 0x_2 + 400x_3 + 900x_4 + 700x_5 \le 245000 \quad (\text{tCO}_2) \quad [y_2 \le 0]$$
3. **Cuota Renovable:**
   $$x_1 + x_2 \ge 312 \iff -x_1 - x_2 \le -312 \quad (\text{GWh}) \quad [y_3 \ge 0]$$
4. **Take-or-pay Gas:**
   $$x_3 \ge 50 \iff -x_3 \le -50 \quad (\text{GWh}) \quad [y_4 \ge 0]$$
5. **Límites de Capacidad:**
   $$0 \le x_1 \le 260 \quad [\mu_1^- \ge 0,\ w_1 \le 0]$$
   $$0 \le x_2 \le 120 \quad [\mu_2^- \ge 0,\ w_2 \le 0]$$
   $$0 \le x_3 \le 300 \quad [\mu_3^- \ge 0,\ w_3 \le 0]$$
   $$0 \le x_4 \le 280 \quad [\mu_4^- \ge 0,\ w_4 \le 0]$$
   $$0 \le x_5 \le 90 \quad [\mu_5^- \ge 0,\ w_5 \le 0]$$

*Nota de notación: $\mu_j^-\ge 0$ es el multiplicador de $x_j\ge 0$ (cota inferior); $w_j = -\mu_j^+ \le 0$ es el multiplicador de $x_j\le \text{cap}_j$ (cota superior), reescrito con signo negativo para que coincida con la convención usada en el dual explícito más abajo (ver Sección 2).*

### 2. Formulación Dual
Sean las variables duales asociadas:
* $y_1$ para la Demanda (igualdad $\to$ no restringida).
* $y_2$ para Emisiones (restricción $\le$ en minimización $\to y_2 \le 0$).
* $y_3$ para Cuota Renovable (restricción $\ge$ en minimización $\to y_3 \ge 0$).
* $y_4$ para Take-or-pay (restricción $\ge$ en minimización $\to y_4 \ge 0$).
* $w_j$ para límites de capacidad superior (restricciones $x_j \le \text{cap}_j \to w_j \le 0$).

**Función Objetivo Dual (Maximizar):**
$$\max g = 780y_1 + 245000y_2 + 312y_3 + 50y_4 + \sum_{j=1}^5 \text{cap}_j w_j$$

**Restricciones Duales (asociadas a variables primales $x_j \ge 0$):**
1. **Para $x_1$:** $y_1 + 0y_2 + y_3 + 0y_4 + w_1 \le 12$
2. **Para $x_2$:** $y_1 + 0y_2 + y_3 + 0y_4 + w_2 \le 8$
3. **Para $x_3$:** $y_1 + 400y_2 + 0y_3 + y_4 + w_3 \le 45$
4. **Para $x_4$:** $y_1 + 900y_2 + 0y_3 + 0y_4 + w_4 \le 32$
5. **Para $x_5$:** $y_1 + 700y_2 + 0y_3 + 0y_4 + w_5 \le 95$

Con $y_1 \in \mathbb{R}$, $y_2 \le 0$, $y_3 \ge 0$, $y_4 \ge 0$, $w_j \le 0$.
```

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog
!pip install pulp -q
import pulp

# --- DATOS PROBLEMA 1 ---
c_p1 = np.array([12, 8, 45, 32, 95])

# Restricciones de desigualdad (A_ub * x <= b_ub)
# 1) Emisiones: 0*x1 + 0*x2 + 400*x3 + 900*x4 + 700*x5 <= 245000
# 2) Cuota renovable: -x1 - x2 <= -312
# 3) Take-or-pay gas: -x3 <= -50
A_ub_p1 = np.array([
    [0, 0, 400, 900, 700],
    [-1, -1, 0, 0, 0],
    [0, 0, -1, 0, 0]
])
b_ub_p1 = np.array([245000, -312, -50])

# Restricciones de igualdad (A_eq * x == b_eq)
# Demanda: x1 + x2 + x3 + x4 + x5 = 780
A_eq_p1 = np.array([[1, 1, 1, 1, 1]])
b_eq_p1 = np.array([780])

# Límites de capacidad
bounds_p1 = [
    (0, 260),  # G1
    (0, 120),  # G2
    (0, 300),  # G3
    (0, 280),  # G4
    (0, 90)    # G5
]

In [2]:
# --- RESOLUCIÓN PRIMAL CON SCIPY ---
res_p1_scipy = linprog(
    c=c_p1,
    A_ub=A_ub_p1,
    b_ub=b_ub_p1,
    A_eq=A_eq_p1,
    b_eq=b_eq_p1,
    bounds=bounds_p1,
    method='highs'
)

print("=== SOLUCIÓN PRIMAL SCIPY (PROB 1) ===")
print(f"Costo óptimo: {res_p1_scipy.fun:.4f} miles USD/semana")
print(f"Generación (G1-G5) GWh: {res_p1_scipy.x}")

=== SOLUCIÓN PRIMAL SCIPY (PROB 1) ===
Costo óptimo: 19870.0000 miles USD/semana
Generación (G1-G5) GWh: [260. 120. 230. 170.   0.]


In [3]:
# --- RESOLUCIÓN PRIMAL CON PULP ---
prob1 = pulp.LpProblem("Despacho_Electrico", pulp.LpMinimize)

x1 = pulp.LpVariable('x_G1', lowBound=0, upBound=260)
x2 = pulp.LpVariable('x_G2', lowBound=0, upBound=120)
x3 = pulp.LpVariable('x_G3', lowBound=0, upBound=300)
x4 = pulp.LpVariable('x_G4', lowBound=0, upBound=280)
x5 = pulp.LpVariable('x_G5', lowBound=0, upBound=90)

vars_p1 = [x1, x2, x3, x4, x5]

# Objetivo
prob1 += 12*x1 + 8*x2 + 45*x3 + 32*x4 + 95*x5

# Restricciones
prob1 += (x1 + x2 + x3 + x4 + x5 == 780, "Demanda")
prob1 += (400*x3 + 900*x4 + 700*x5 <= 245000, "Emisiones")
prob1 += (x1 + x2 >= 312, "Cuota_Renovable")
prob1 += (x3 >= 50, "Take_or_pay")

prob1.solve(pulp.PULP_CBC_CMD(msg=False))

print("=== SOLUCIÓN PRIMAL PULP (PROB 1) ===")
print(f"Estado: {pulp.LpStatus[prob1.status]}")
print(f"Costo óptimo: {pulp.value(prob1.objective):.4f} miles USD/semana")
for v in vars_p1:
    print(f"{v.name}: {v.varValue:.4f} GWh")

=== SOLUCIÓN PRIMAL PULP (PROB 1) ===
Estado: Optimal
Costo óptimo: 19870.0000 miles USD/semana
x_G1: 260.0000 GWh
x_G2: 120.0000 GWh
x_G3: 230.0000 GWh
x_G4: 170.0000 GWh
x_G5: 0.0000 GWh


In [4]:
# --- RESOLUCIÓN DEL DUAL DIRECTO COMO UN PL INDEPENDIENTE (SCIPY) ---
# Queremos maximizar la f.obj dual. Para linprog minimizamos el negativo.
# Variables duales: y1 (libre), y2 (<=0), y3 (>=0), y4 (>=0), w1..w5 (<=0)
# x = [y1, y2, y3, y4, w1, w2, w3, w4, w5]

c_dual = -np.array([780, 245000, 312, 50, 260, 120, 300, 280, 90])

# Restricciones duales (todas son <=)
A_ub_dual = np.array([
    [1, 0, 1, 0, 1, 0, 0, 0, 0], # G1
    [1, 0, 1, 0, 0, 1, 0, 0, 0], # G2
    [1, 400, 0, 1, 0, 0, 1, 0, 0], # G3
    [1, 900, 0, 0, 0, 0, 0, 1, 0], # G4
    [1, 700, 0, 0, 0, 0, 0, 0, 1]  # G5
])
b_ub_dual = np.array([12, 8, 45, 32, 95])

bounds_dual = [
    (None, None), # y1
    (None, 0),    # y2
    (0, None),    # y3
    (0, None),    # y4
    (None, 0),    # w1
    (None, 0),    # w2
    (None, 0),    # w3
    (None, 0),    # w4
    (None, 0)     # w5
]

res_p1_dual = linprog(c=c_dual, A_ub=A_ub_dual, b_ub=b_ub_dual, bounds=bounds_dual, method='highs')
print("=== SOLUCIÓN DUAL DIRECTO (PROB 1) ===")
print(f"Objetivo dual óptimo: {-res_p1_dual.fun:.4f} miles USD/semana")
names_dual = ['y_Demanda', 'y_Emisiones', 'y_CuotaRenov', 'y_TakeOrPay', 'w_G1', 'w_G2', 'w_G3', 'w_G4', 'w_G5']
for name, val in zip(names_dual, res_p1_dual.x):
    print(f"{name}: {val:.6f}")

=== SOLUCIÓN DUAL DIRECTO (PROB 1) ===
Objetivo dual óptimo: 19870.0000 miles USD/semana
y_Demanda: 55.400000
y_Emisiones: -0.026000
y_CuotaRenov: 0.000000
y_TakeOrPay: 0.000000
w_G1: -43.400000
w_G2: -47.400000
w_G3: 0.000000
w_G4: 0.000000
w_G5: 0.000000


In [5]:
# --- VERIFICACIÓN DE DUALIDAD FUERTE (PROB 1) ---
primal_val = res_p1_scipy.fun
dual_val = -res_p1_dual.fun
gap_abs = abs(primal_val - dual_val)
gap_rel = gap_abs / (abs(primal_val) + 1e-12)
print(f"Gap Absoluto: {gap_abs:.4e}")
print(f"Gap Relativo: {gap_rel:.4e}")

Gap Absoluto: 0.0000e+00
Gap Relativo: 0.0000e+00


In [6]:
# === CONVERSIÓN DE UNIDADES EN LOS REPORTES (Problema 1) ===
# El PDF exige reportar todo valor con sus unidades y advierte explícitamente que el
# dual de emisiones sale en "miles USD/tCO2" y debe convertirse a USD/tCO2.

y_emisiones_miles_usd = res_p1_dual.x[1]          # tal como sale del modelo (miles USD/tCO2)
y_emisiones_usd = y_emisiones_miles_usd * 1000    # USD/tCO2

print(f"Z* = {res_p1_scipy.fun:.1f} miles USD/semana  =  {res_p1_scipy.fun/1000:.3f} millones USD/semana")
print(f"y_Emisiones = {y_emisiones_miles_usd:.4f} miles USD/tCO2  =  {y_emisiones_usd:.2f} USD/tCO2")
# Valor de referencia del propio PDF del taller: Z*≈19.87 millones USD, dual emisiones ≈ -26 USD/tCO2

Z* = 19870.0 miles USD/semana  =  19.870 millones USD/semana
y_Emisiones = -0.0260 miles USD/tCO2  =  -26.00 USD/tCO2


### Pregunta 4 (comparación explícita) y Pregunta 5 (costos reducidos) — Problema 1

In [7]:
nombres_restr_p1 = ["Emisiones(<=)", "Renovable(<=,neg)", "TakeOrPay(<=,neg)"]
comparacion_p1 = pd.DataFrame({
    "Restricción": nombres_restr_p1,
    "scipy ineqlin.marginals (forma <=)": res_p1_scipy.ineqlin.marginals,
    "Dual explícito (signo económico)": [res_p1_dual.x[1], -res_p1_dual.x[2], -res_p1_dual.x[3]],
    "¿Coinciden en magnitud?": np.isclose(
        np.abs(res_p1_scipy.ineqlin.marginals),
        np.abs([res_p1_dual.x[1], res_p1_dual.x[2], res_p1_dual.x[3]])
    )
})
print("Nota de signo: las filas 'Renovable' y 'TakeOrPay' se negaron al construir A_ub_p1")
print("(de x>=b a -x<=-b), por eso ineqlin.marginals sale con signo opuesto al dual")
print("'económico' y3,y4>=0 que definimos nosotros. La MAGNITUD debe coincidir (error frecuente #5 del anexo).\n")
display(comparacion_p1)

print("Demanda (igualdad): scipy=", res_p1_scipy.eqlin.marginals[0], " vs dual explícito y1=", res_p1_dual.x[0])

# --- Costos reducidos (Pregunta 5) ---
costos_reducidos_p1 = pd.DataFrame({
    "Planta": ["G1","G2","G3","G4","G5"],
    "x_j*": res_p1_scipy.x,
    "Costo reducido (lower, activo si x_j=0)": res_p1_scipy.lower.marginals,
    "Costo reducido (upper, activo si x_j=cap_j)": res_p1_scipy.upper.marginals,
    "¿Restricción dual activa?": (np.abs(res_p1_scipy.lower.marginals) > 1e-9) | (np.abs(res_p1_scipy.upper.marginals) > 1e-9)
})
display(costos_reducidos_p1)

Nota de signo: las filas 'Renovable' y 'TakeOrPay' se negaron al construir A_ub_p1
(de x>=b a -x<=-b), por eso ineqlin.marginals sale con signo opuesto al dual
'económico' y3,y4>=0 que definimos nosotros. La MAGNITUD debe coincidir (error frecuente #5 del anexo).



,Restricción,scipy ineqlin.marginals (forma <=),Dual explícito (signo económico),¿Coinciden en magnitud?
0,Emisiones(<=),-0.026,-0.026,True
1,"Renovable(<=,neg)",-0.000,-0.000,True
2,"TakeOrPay(<=,neg)",-0.000,-0.000,True


Demanda (igualdad): scipy= 55.4  vs dual explícito y1= 55.4


,Planta,x_j*,"Costo reducido (lower, activo si x_j=0)","Costo reducido (upper, activo si x_j=cap_j)",¿Restricción dual activa?
0,G1,260.0,0.0,-43.4,True
1,G2,120.0,0.0,-47.4,True
2,G3,230.0,0.0,0.0,False
3,G4,170.0,0.0,0.0,False
4,G5,0.0,57.8,0.0,True



## PROBLEMA 2: Portafolio de Crédito

### 1. Formulación Primal (Matemática)
Sea $x_j$ el monto colocado en la línea de crédito $j$ en millones USD ($j \in \{1, 2, 3, 4, 5\}$ para Consumo, Vehículo, Hipotecario, PyME, Corporativo).
Convertimos el rendimiento nominal a fracción para los cálculos.

**Función Objetivo (Maximizar Rendimiento Neto en MUSD):**
$$\max z = 0.165x_1 + 0.115x_2 + 0.070x_3 + 0.130x_4 + 0.060x_5$$

**Restricciones:**
1. **Presupuesto:**
   $$x_1 + x_2 + x_3 + x_4 + x_5 = 600 \quad (\text{MUSD}) \quad [y_1 \in \mathbb{R}]$$
2. **Apetito de Riesgo:**
   $$9x_1 + 5x_2 + 2x_3 + 6x_4 + 1x_5 \le 3300 \quad (\text{puntos}) \quad [y_2 \ge 0]$$
3. **Regulatorio Bajo Riesgo:**
   $$x_3 + x_5 \ge 120 \iff -x_3 - x_5 \le -120 \quad (\text{MUSD}) \quad [y_3 \le 0]$$
4. **Meta Comercial PyME:**
   $$x_4 \ge 50 \iff -x_4 \le -50 \quad (\text{MUSD}) \quad [y_4 \le 0]$$
5. **Límites de Concentración:**
   $$0 \le x_1 \le 150$$
   $$0 \le x_2 \le 120$$
   $$0 \le x_3 \le 250$$
   $$0 \le x_4 \le 180$$
   $$0 \le x_5 \le 250$$
```

In [8]:
# --- DATOS PROBLEMA 2 ---
c_p2 = -np.array([0.165, 0.115, 0.070, 0.130, 0.060]) # Negativo para maximizar con scipy

# Desigualdades (A_ub * x <= b_ub)
A_ub_p2 = np.array([
    [9, 5, 2, 6, 1],
    [0, 0, -1, 0, -1],
    [0, 0, 0, -1, 0]
])
b_ub_p2 = np.array([3300, -120, -50])

# Igualdad
A_eq_p2 = np.array([[1, 1, 1, 1, 1]])
b_eq_p2 = np.array([600])

bounds_p2 = [
    (0, 150),
    (0, 120),
    (0, 250),
    (0, 180),
    (0, 250)
]

In [9]:
# --- RESOLUCIÓN PRIMAL CON SCIPY (PROB 2) ---
res_p2_scipy = linprog(
    c=c_p2,
    A_ub=A_ub_p2,
    b_ub=b_ub_p2,
    A_eq=A_eq_p2,
    b_eq=b_eq_p2,
    bounds=bounds_p2,
    method='highs'
)

print("=== SOLUCIÓN PRIMAL SCIPY (PROB 2) ===")
print(f"Rendimiento neto óptimo: {-res_p2_scipy.fun:.4f} MUSD")
print(f"Asignación líneas (L1-L5) MUSD: {res_p2_scipy.x}")

=== SOLUCIÓN PRIMAL SCIPY (PROB 2) ===
Rendimiento neto óptimo: 72.1500 MUSD
Asignación líneas (L1-L5) MUSD: [150. 120. 120. 180.  30.]


In [10]:
# --- RESOLUCIÓN PRIMAL CON PULP (PROB 2) ---
prob2 = pulp.LpProblem("Portafolio_Credito", pulp.LpMaximize)

l1 = pulp.LpVariable('x_L1', lowBound=0, upBound=150)
l2 = pulp.LpVariable('x_L2', lowBound=0, upBound=120)
l3 = pulp.LpVariable('x_L3', lowBound=0, upBound=250)
l4 = pulp.LpVariable('x_L4', lowBound=0, upBound=180)
l5 = pulp.LpVariable('x_L5', lowBound=0, upBound=250)

vars_p2 = [l1, l2, l3, l4, l5]

prob2 += 0.165*l1 + 0.115*l2 + 0.070*l3 + 0.130*l4 + 0.060*l5

prob2 += (l1 + l2 + l3 + l4 + l5 == 600, "Presupuesto")
prob2 += (9*l1 + 5*l2 + 2*l3 + 6*l4 + 1*l5 <= 3300, "Apetito_Riesgo")
prob2 += (l3 + l5 >= 120, "Bajo_Riesgo")
prob2 += (l4 >= 50, "Meta_PyME")

prob2.solve(pulp.PULP_CBC_CMD(msg=False))

print("=== SOLUCIÓN PRIMAL PULP (PROB 2) ===")
print(f"Rendimiento neto óptimo: {pulp.value(prob2.objective):.4f} MUSD")
for v in vars_p2:
    print(f"{v.name}: {v.varValue:.4f} MUSD")

=== SOLUCIÓN PRIMAL PULP (PROB 2) ===
Rendimiento neto óptimo: 72.1500 MUSD
x_L1: 150.0000 MUSD
x_L2: 120.0000 MUSD
x_L3: 120.0000 MUSD
x_L4: 180.0000 MUSD
x_L5: 30.0000 MUSD


In [11]:
# --- RESOLUCIÓN DEL DUAL DIRECTO (PROB 2) ---
# Variables duales: y1 (libre), y2 (>=0), y3 (<=0), y4 (<=0), w1..w5 (>=0)
# Buscamos minimizar g_dual = 600y1 + 3300y2 + 120y3 + 50y4 + sum(lim_j * w_j)
c_dual_p2 = np.array([600, 3300, 120, 50, 150, 120, 250, 180, 250])

A_ub_dual_p2 = np.array([
    [1, 9, 0, 0, 1, 0, 0, 0, 0],  # L1 (1y1 + 9y2 + w1 >= c1)
    [1, 5, 0, 0, 0, 1, 0, 0, 0],  # L2 (1y1 + 5y2 + w2 >= c2)
    [1, 2, 1, 0, 0, 0, 1, 0, 0],  # L3 (1y1 + 2y2 + 1y3 + w3 >= c3) <-- Corrected y3 coeff
    [1, 6, 0, 1, 0, 0, 0, 1, 0],  # L4 (1y1 + 6y2 + 1y4 + w4 >= c4) <-- Corrected y4 coeff
    [1, 1, 1, 0, 0, 0, 0, 0, 1]   # L5 (1y1 + 1y2 + 1y3 + w5 >= c5) <-- Corrected y3 coeff
])
b_ub_dual_p2 = np.array([0.165, 0.115, 0.070, 0.130, 0.060])

bounds_dual_p2 = [
    (None, None), # y1
    (0, None),    # y2
    (None, 0),    # y3
    (None, 0),    # y4
    (0, None),    # w1
    (0, None),    # w2
    (0, None),    # w3
    (0, None),    # w4
    (0, None)     # w5
]

res_p2_dual = linprog(c=c_dual_p2, A_ub=-A_ub_dual_p2, b_ub=-b_ub_dual_p2, bounds=bounds_dual_p2, method='highs')
print("=== SOLUCIÓN DUAL DIRECTO (PROB 2) ===")

if res_p2_dual.success:
    print(f"Objetivo dual óptimo: {res_p2_dual.fun:.6f} MUSD")
    names_dual_p2 = ['y_Presupuesto', 'y_ApetitoRiesgo', 'y_BajoRiesgo', 'y_MetaPyME', 'w_L1', 'w_L2', 'w_L3', 'w_L4', 'w_L5']
    for name, val in zip(names_dual_p2, res_p2_dual.x):
        print(f"{name}: {val:.6f}")
else:
    print(f"Linprog no encontró una solución óptima.")
    print(f"Status: {res_p2_dual.status}")
    print(f"Message: {res_p2_dual.message}")

=== SOLUCIÓN DUAL DIRECTO (PROB 2) ===
Objetivo dual óptimo: 72.150000 MUSD
y_Presupuesto: 0.050000
y_ApetitoRiesgo: 0.010000
y_BajoRiesgo: 0.000000
y_MetaPyME: 0.000000
w_L1: 0.025000
w_L2: 0.015000
w_L3: 0.000000
w_L4: 0.020000
w_L5: 0.000000


### Verificación de dualidad fuerte — Problema 2

In [12]:
primal_val_p2 = -res_p2_scipy.fun   # negativo porque c_p2 se negó para maximizar
dual_val_p2 = res_p2_dual.fun
gap_abs_p2 = abs(primal_val_p2 - dual_val_p2)
gap_rel_p2 = gap_abs_p2 / (abs(primal_val_p2) + 1e-12)
print(f"Gap Absoluto P2: {gap_abs_p2:.4e}")
print(f"Gap Relativo P2: {gap_rel_p2:.4e}")

Gap Absoluto P2: 0.0000e+00
Gap Relativo P2: 0.0000e+00


### Pregunta 4 (comparación explícita) y Pregunta 5 (costos reducidos) — Problema 2

In [13]:
comparacion_p2 = pd.DataFrame({
    "Restricción": ["ApetitoRiesgo(<=)", "BajoRiesgo(<=,neg)", "MetaPyME(<=,neg)"],
    "scipy ineqlin.marginals (problema min(-c))": res_p2_scipy.ineqlin.marginals,
    "Dual explícito (signo económico)": [res_p2_dual.x[1], res_p2_dual.x[2], res_p2_dual.x[3]],
    "¿Coinciden en magnitud?": np.isclose(
        np.abs(res_p2_scipy.ineqlin.marginals),
        np.abs([res_p2_dual.x[1], res_p2_dual.x[2], res_p2_dual.x[3]])
    )
})
display(comparacion_p2)
print("Presupuesto (igualdad): scipy=", res_p2_scipy.eqlin.marginals[0], " vs dual explícito y1=", res_p2_dual.x[0])
print("Nota: aquí hay DOS fuentes de cambio de signo respecto a los datos crudos de scipy:")
print("(1) c_p2 se negó para pasar de max a min, invirtiendo el signo de TODOS los duales;")
print("(2) las filas de BajoRiesgo y MetaPyME también se negaron para pasar de >= a <=")
print("    (esta segunda inversión se cancela con la primera para esas dos filas).")

costos_reducidos_p2 = pd.DataFrame({
    "Línea": ["L1","L2","L3","L4","L5"],
    "x_j*": res_p2_scipy.x,
    "Costo reducido (lower)": res_p2_scipy.lower.marginals,
    "Costo reducido (upper)": res_p2_scipy.upper.marginals,
    "¿Restricción dual activa?": (np.abs(res_p2_scipy.lower.marginals) > 1e-9) | (np.abs(res_p2_scipy.upper.marginals) > 1e-9)
})
display(costos_reducidos_p2)

,Restricción,scipy ineqlin.marginals (problema min(-c)),Dual explícito (signo económico),¿Coinciden en magnitud?
0,ApetitoRiesgo(<=),-0.01,0.01,True
1,"BajoRiesgo(<=,neg)",-0.00,0.00,True
2,"MetaPyME(<=,neg)",-0.00,0.00,True


Presupuesto (igualdad): scipy= -0.04999999999999999  vs dual explícito y1= 0.04999999999999999
Nota: aquí hay DOS fuentes de cambio de signo respecto a los datos crudos de scipy:
(1) c_p2 se negó para pasar de max a min, invirtiendo el signo de TODOS los duales;
(2) las filas de BajoRiesgo y MetaPyME también se negaron para pasar de >= a <=
    (esta segunda inversión se cancela con la primera para esas dos filas).


,Línea,x_j*,Costo reducido (lower),Costo reducido (upper),¿Restricción dual activa?
0,L1,150.0,0.0,-0.025,True
1,L2,120.0,0.0,-0.015,True
2,L3,120.0,0.0,0.000,False
3,L4,180.0,0.0,-0.020,True
4,L5,30.0,0.0,0.000,False


## SECCIÓN V: Verificación crítica (transversal a los dos problemas)

### V.1 Trazabilidad modelo <-> código y consistencia dimensional

In [14]:
# Sin esta tabla no se puede verificar que cada término del código corresponde
# a un término del enunciado (criterio V.1, 4 pts).

trazabilidad_p1 = pd.DataFrame([
    ["Costo variable G_j", "c_j x_j", "c_p1[j]", "miles USD/GWh · GWh = miles USD", "min"],
    ["Demanda semanal", "Σx_j = 780", "A_eq_p1, b_eq_p1", "GWh", "="],
    ["Tope de emisiones", "Σe_j x_j ≤ 245000", "A_ub_p1[0], b_ub_p1[0]", "tCO2/GWh · GWh = tCO2", "≤"],
    ["Cuota renovable", "x1+x2 ≥ 0.4·780", "A_ub_p1[1], b_ub_p1[1]", "GWh", "≥"],
    ["Take-or-pay gas", "x3 ≥ 50", "A_ub_p1[2], b_ub_p1[2]", "GWh", "≥"],
    ["Capacidad planta j", "0 ≤ x_j ≤ cap_j", "bounds_p1[j]", "GWh", "cota"],
], columns=["Elemento del enunciado","Expresión algebraica","Objeto en el código","Unidades","Sentido"])
print("--- Trazabilidad Problema 1 ---")
display(trazabilidad_p1)

trazabilidad_p2 = pd.DataFrame([
    ["Rendimiento neto línea j", "r_j x_j", "c_p2[j] (negado)", "fracción · MUSD = MUSD", "max"],
    ["Presupuesto", "Σx_j = 600", "A_eq_p2, b_eq_p2", "MUSD", "="],
    ["Apetito de riesgo", "Σriesgo_j x_j ≤ 3300", "A_ub_p2[0], b_ub_p2[0]", "pts/millón · MUSD = pts", "≤"],
    ["Bajo riesgo (regulatorio)", "x3+x5 ≥ 0.20·600", "A_ub_p2[1], b_ub_p2[1]", "MUSD", "≥"],
    ["Meta comercial PyME", "x4 ≥ 50", "A_ub_p2[2], b_ub_p2[2]", "MUSD", "≥"],
    ["Límite concentración j", "0 ≤ x_j ≤ límite_j", "bounds_p2[j]", "MUSD", "cota"],
], columns=["Elemento del enunciado","Expresión algebraica","Objeto en el código","Unidades","Sentido"])
print("\n--- Trazabilidad Problema 2 ---")
display(trazabilidad_p2)

# Verificación dimensional explícita (numérica, no solo declarativa)
e = np.array([0,0,400,900,700])            # tCO2/GWh
x_test = res_p1_scipy.x                     # GWh
emisiones_tCO2 = e @ x_test                 # tCO2/GWh * GWh = tCO2
print(f"\nChequeo dimensional P1: {e[2]} tCO2/GWh * {x_test[2]:.1f} GWh = {e[2]*x_test[2]:.1f} tCO2 (unidad correcta)")
print(f"Emisiones totales: {emisiones_tCO2:.1f} tCO2 (debe ser ≤ 245000)")

r_pct = np.array([16.5, 11.5, 7.0, 13.0, 6.0])
r_frac = r_pct / 100
print(f"\nChequeo dimensional P2: {r_pct[0]}%  ->  {r_frac[0]} (fracción), usado en c_p2")
assert np.allclose(r_frac, np.array([0.165,0.115,0.070,0.130,0.060])), "Debe coincidir con c_p2 usado en el código"
print("OK: la conversión de % a fracción es consistente con c_p2.")

--- Trazabilidad Problema 1 ---


,Elemento del enunciado,Expresión algebraica,Objeto en el código,Unidades,Sentido
0,Costo variable G_j,c_j x_j,c_p1[j],miles USD/GWh · GWh = miles USD,min
1,Demanda semanal,Σx_j = 780,"A_eq_p1, b_eq_p1",GWh,=
2,Tope de emisiones,Σe_j x_j ≤ 245000,"A_ub_p1[0], b_ub_p1[0]",tCO2/GWh · GWh = tCO2,≤
3,Cuota renovable,x1+x2 ≥ 0.4·780,"A_ub_p1[1], b_ub_p1[1]",GWh,≥
4,Take-or-pay gas,x3 ≥ 50,"A_ub_p1[2], b_ub_p1[2]",GWh,≥
5,Capacidad planta j,0 ≤ x_j ≤ cap_j,bounds_p1[j],GWh,cota



--- Trazabilidad Problema 2 ---


,Elemento del enunciado,Expresión algebraica,Objeto en el código,Unidades,Sentido
0,Rendimiento neto línea j,r_j x_j,c_p2[j] (negado),fracción · MUSD = MUSD,max
1,Presupuesto,Σx_j = 600,"A_eq_p2, b_eq_p2",MUSD,=
2,Apetito de riesgo,Σriesgo_j x_j ≤ 3300,"A_ub_p2[0], b_ub_p2[0]",pts/millón · MUSD = pts,≤
3,Bajo riesgo (regulatorio),x3+x5 ≥ 0.20·600,"A_ub_p2[1], b_ub_p2[1]",MUSD,≥
4,Meta comercial PyME,x4 ≥ 50,"A_ub_p2[2], b_ub_p2[2]",MUSD,≥
5,Límite concentración j,0 ≤ x_j ≤ límite_j,bounds_p2[j],MUSD,cota



Chequeo dimensional P1: 400 tCO2/GWh * 230.0 GWh = 92000.0 tCO2 (unidad correcta)
Emisiones totales: 245000.0 tCO2 (debe ser ≤ 245000)

Chequeo dimensional P2: 16.5%  ->  0.165 (fracción), usado en c_p2
OK: la conversión de % a fracción es consistente con c_p2.


### V.2 Verificación de Factibilidad Primal sin Solver (Residuos)

In [27]:
# --- VERIFICACIÓN DE RESIDUOS PRIMALES CON NUMPY ---
tol = 1e-6  # Tolerancia numérica estándar para aceptar aproximaciones a cero

def verificar_factibilidad_sin_solver(nombre, x, A_eq, b_eq, A_ub, b_ub, bounds, tol=1e-6):
    """
    Evalúa la viabilidad técnica de una solución (vector x) sin utilizar solvers externos,
    calculando las holguras físicas y contrastándolas contra la tolerancia matemática.
    """
    print(f"--- {nombre} ---")
    ok = True

    # 1. Validación de restricciones de igualdad (Ej: Demanda / Presupuesto)
    # Se calcula la diferencia: A_eq * x - b_eq. El residuo absoluto debe ser menor a tol.
    residuos_eq = A_eq @ x - b_eq
    for i, r in enumerate(residuos_eq):
        estado = abs(r) < tol
        ok &= estado
        print(f"Igualdad {i}: residuo={r:.2e}  -> {'OK' if estado else 'FALLA'} (tol={tol:.0e})")

    # 2. Validación de restricciones de desigualdad (Ej: Emisiones, Límites regulatorios)
    # Para restricciones de la forma A_ub * x <= b_ub, el valor (A_ub*x - b_ub) debe ser <= 0.
    # Con tolerancia numérica, permitimos que supere ligeramente hasta un valor menor a tol.
    residuos_ub = A_ub @ x - b_ub
    for i, r in enumerate(residuos_ub):
        estado = r < tol
        ok &= estado
        print(f"Desigualdad {i}: A_ub@x - b_ub = {r:.4f}  -> {'OK (<=0)' if estado else 'FALLA'} (tol={tol:.0e})")

    # 3. Validación de No-Negatividad (x >= 0)
    # Asegura que no existan asignaciones o producciones con signo negativo físico.
    no_negativo = np.all(x >= -tol)
    ok &= no_negativo
    print(f"No negatividad: min(x)={x.min():.4f}  -> {'OK' if no_negativo else 'FALLA'}")

    # 4. Validación de cotas superiores (x <= cap)
    # Extrae el límite superior de cada variable y comprueba que ninguna variable lo exceda.
    ub_arr = np.array([b[1] for b in bounds])
    dentro_cota = np.all(x <= ub_arr + tol)
    ok &= dentro_cota
    print(f"Cotas superiores: max(x - cap)={np.max(x - ub_arr):.4f}  -> {'OK' if dentro_cota else 'FALLA'}")

    print(f"=> Factibilidad primal {nombre}: {'CUMPLE' if ok else 'NO CUMPLE'}\n")
    return ok

ok_fact_p1 = verificar_factibilidad_sin_solver("Problema 1", res_p1_scipy.x, A_eq_p1, b_eq_p1, A_ub_p1, b_ub_p1, bounds_p1, tol)
ok_fact_p2 = verificar_factibilidad_sin_solver("Problema 2", res_p2_scipy.x, A_eq_p2, b_eq_p2, A_ub_p2, b_ub_p2, bounds_p2, tol)

--- Problema 1 ---
Igualdad 0: residuo=0.00e+00  -> OK (tol=1e-06)
Desigualdad 0: A_ub@x - b_ub = 0.0000  -> OK (<=0) (tol=1e-06)
Desigualdad 1: A_ub@x - b_ub = -68.0000  -> OK (<=0) (tol=1e-06)
Desigualdad 2: A_ub@x - b_ub = -180.0000  -> OK (<=0) (tol=1e-06)
No negatividad: min(x)=0.0000  -> OK
Cotas superiores: max(x - cap)=0.0000  -> OK
=> Factibilidad primal Problema 1: CUMPLE

--- Problema 2 ---
Igualdad 0: residuo=0.00e+00  -> OK (tol=1e-06)
Desigualdad 0: A_ub@x - b_ub = 0.0000  -> OK (<=0) (tol=1e-06)
Desigualdad 1: A_ub@x - b_ub = -30.0000  -> OK (<=0) (tol=1e-06)
Desigualdad 2: A_ub@x - b_ub = -130.0000  -> OK (<=0) (tol=1e-06)
No negatividad: min(x)=30.0000  -> OK
Cotas superiores: max(x - cap)=0.0000  -> OK
=> Factibilidad primal Problema 2: CUMPLE



### V.4 Verificación de dualidad fuerte (consolidado, ambos problemas)

In [16]:
print("--- Problema 1 ---")
print(f"Primal={primal_val:.4f}, Dual={dual_val:.4f}, gap_abs={gap_abs:.2e}, gap_rel={gap_rel:.2e}")
print("--- Problema 2 ---")
print(f"Primal={primal_val_p2:.4f}, Dual={dual_val_p2:.4f}, gap_abs={gap_abs_p2:.2e}, gap_rel={gap_rel_p2:.2e}")
print("\nAmbos gaps relativos deben ser < 1e-6 para aceptar la solución dual como correcta.")

--- Problema 1 ---
Primal=19870.0000, Dual=19870.0000, gap_abs=0.00e+00, gap_rel=0.00e+00
--- Problema 2 ---
Primal=72.1500, Dual=72.1500, gap_abs=0.00e+00, gap_rel=0.00e+00

Ambos gaps relativos deben ser < 1e-6 para aceptar la solución dual como correcta.


### V.5 Verificación de holgura complementaria y factibilidad dual (consolidada)

In [30]:
def verificar_holgura_complementaria(nombre, res_primal, res_dual_explicito, A_ub, b_ub, bounds, tol=1e-6):
    """
    Audita las condiciones de Karush-Kuhn-Tucker (KHT) sobre holgura complementaria.
    Verifica que el producto de los recursos libres (holgura) por su costo de oportunidad (variable dual) sea cero.
    """
    print(f"--- {nombre}: y_i * s_i = 0 (restricciones) ---")
    x = res_primal.x
    # s_i son los márgenes o excesos de los recursos (slacks primales)
    slacks = b_ub - A_ub @ x
    # Tomamos el valor absoluto de las variables duales obtenidas del solver
    y_ineq = np.abs(res_primal.ineqlin.marginals)
    ok_restr = True

    # Para cada restricción, validamos que si s_i > 0, entonces y_i == 0 (y viceversa)
    for i, (s, y) in enumerate(zip(slacks, y_ineq)):
        prod = s * y
        estado = abs(prod) < tol
        ok_restr &= estado
        print(f"  Restr. {i}: holgura={s:.4f}, |dual|={y:.6f}, y*s={prod:.2e}  -> {'OK' if estado else 'FALLA'}")

    # Validación de holgura complementaria en variables y cotas:
    # - Si una variable x_j está libre entre sus límites (básica), su costo reducido debe ser cero.
    # - Si x_j está fija en su cota inferior (x_j = lb), su holgura inferior es cero.
    # - Si x_j está en su cota superior (x_j = ub), la holgura superior (ub - x_j) es cero.
    print(f"--- {nombre}: (cota - x_j) * costo_reducido_j = 0 (variables) ---")
    ok_var = True
    for j, xj in enumerate(x):
        lb, ub = bounds[j]
        if abs(xj - lb) < 1e-6: # Variable está en el límite inferior
            cr, slack_var, cota = res_primal.lower.marginals[j], xj - lb, "inferior"
        elif abs(xj - ub) < 1e-6: # Variable topó en el límite superior
            cr, slack_var, cota = res_primal.upper.marginals[j], ub - xj, "superior"
        else: # Variable básica o intermedia libre
            cr, slack_var, cota = 0.0, 0.0, "ninguna (básica)"

        prod = slack_var * cr
        estado = abs(prod) < tol
        ok_var &= estado
        print(f"  x_{j+1}={xj:.4f} (cota {cota}), costo_red={cr:.6f}, slack*cr={prod:.2e}  -> {'OK' if estado else 'FALLA'}")
    return ok_restr and ok_var

ok_p1 = verificar_holgura_complementaria("PROBLEMA 1", res_p1_scipy, res_p1_dual, A_ub_p1, b_ub_p1, bounds_p1)
print()
ok_p2 = verificar_holgura_complementaria("PROBLEMA 2", res_p2_scipy, res_p2_dual, A_ub_p2, b_ub_p2, bounds_p2)
print(f"\nHolgura complementaria: P1={'CUMPLE' if ok_p1 else 'NO CUMPLE'}, P2={'CUMPLE' if ok_p2 else 'NO CUMPLE'}")

print("\n--- Factibilidad dual: signo de cada variable dual segun el sentido de su restriccion ---")
print(f"P1 y_Emisiones (<=)  = {res_p1_dual.x[1]:.4f}  (debe ser <=0): {res_p1_dual.x[1]<=1e-9}")
print(f"P1 y_CuotaRenov (>=) = {res_p1_dual.x[2]:.4f}  (debe ser >=0): {res_p1_dual.x[2]>=-1e-9}")
print(f"P1 y_TakeOrPay (>=)  = {res_p1_dual.x[3]:.4f}  (debe ser >=0): {res_p1_dual.x[3]>=-1e-9}")
print(f"P2 y_ApetitoRiesgo (<=) = {res_p2_dual.x[1]:.4f}  (debe ser >=0 en signo economico): {res_p2_dual.x[1]>=-1e-9}")
print(f"P2 y_BajoRiesgo (>=)    = {res_p2_dual.x[2]:.4f}  (debe ser <=0 en signo economico): {res_p2_dual.x[2]<=1e-9}")
print(f"P2 y_MetaPyME (>=)      = {res_p2_dual.x[3]:.4f}  (debe ser <=0 en signo economico): {res_p2_dual.x[3]<=1e-9}")

--- PROBLEMA 1: y_i * s_i = 0 (restricciones) ---
  Restr. 0: holgura=0.0000, |dual|=0.026000, y*s=0.00e+00  -> OK
  Restr. 1: holgura=68.0000, |dual|=0.000000, y*s=0.00e+00  -> OK
  Restr. 2: holgura=180.0000, |dual|=0.000000, y*s=0.00e+00  -> OK
--- PROBLEMA 1: (cota - x_j) * costo_reducido_j = 0 (variables) ---
  x_1=260.0000 (cota superior), costo_red=-43.400000, slack*cr=-0.00e+00  -> OK
  x_2=120.0000 (cota superior), costo_red=-47.400000, slack*cr=-0.00e+00  -> OK
  x_3=230.0000 (cota ninguna (básica)), costo_red=0.000000, slack*cr=0.00e+00  -> OK
  x_4=170.0000 (cota ninguna (básica)), costo_red=0.000000, slack*cr=0.00e+00  -> OK
  x_5=0.0000 (cota inferior), costo_red=57.800000, slack*cr=0.00e+00  -> OK

--- PROBLEMA 2: y_i * s_i = 0 (restricciones) ---
  Restr. 0: holgura=0.0000, |dual|=0.010000, y*s=0.00e+00  -> OK
  Restr. 1: holgura=30.0000, |dual|=0.000000, y*s=0.00e+00  -> OK
  Restr. 2: holgura=130.0000, |dual|=0.000000, y*s=0.00e+00  -> OK
--- PROBLEMA 2: (cota - x_j) 

### V.6 Precios sombra por perturbación finita (con detección de punto de quiebre)

In [29]:
def encontrar_rango_precio_sombra(nombre_recurso, resolver_fn, b0, y_shadow, paso=1.0, max_iter=1000, tol=1e-4):
    """
    Calcula experimentalmente el rango de estabilidad de la base óptima actual.
    Aumenta y disminuye el recurso b0 de forma iterativa y compara el nuevo valor óptimo real Z(b0+d)
    frente a la aproximación teórica del gradiente (Z0 + y*d). Cuando se desvían, detecta un cambio de base (quiebre).
    """
    Z0 = resolver_fn(b0)

    # 1. Exploración hacia arriba (incremento del recurso b0)
    delta_up = 0
    while delta_up < max_iter*paso:
        delta_up += paso
        Z1 = resolver_fn(b0 + delta_up)
        pred = Z0 + y_shadow*delta_up
        # Si el modelo se vuelve infactible o la aproximación lineal se rompe, salimos del bucle
        if Z1 == np.inf or abs(Z1 - pred) > tol:
            break

    # 2. Exploración hacia abajo (reducción del recurso b0)
    delta_dn = 0
    while delta_dn < max_iter*paso:
        delta_dn += paso
        Z1 = resolver_fn(b0 - delta_dn)
        pred = Z0 - y_shadow*delta_dn
        # Detenerse si hay infactibilidad o cambio de pendientes (nueva base óptima)
        if Z1 == np.inf or abs(Z1 - pred) > tol:
            break

    # El rango válido de b0 se sitúa entre el último punto que funcionó bien hacia abajo y hacia arriba
    lo, hi = b0-delta_dn+paso, b0+delta_up-paso
    print(f"{nombre_recurso}: precio sombra={y_shadow:.4f}, rango de validez del RHS ≈ [{lo:.1f}, {hi:.1f}]")
    return lo, hi

# --- Recurso 1 de detalle, Problema 1: Emisiones ---
def Z_emisiones_p1(cap_emisiones):
    # Re-optimiza el problema 1 modificando únicamente el límite de emisiones (LHS de la restricción 0)
    r = linprog(c=c_p1, A_ub=A_ub_p1, b_ub=[cap_emisiones, -312, -50],
                A_eq=A_eq_p1, b_eq=b_eq_p1, bounds=bounds_p1, method='highs')
    return r.fun if r.success else np.inf

rango_emisiones = encontrar_rango_precio_sombra("Emisiones (P1)", Z_emisiones_p1, 245000, res_p1_dual.x[1], paso=500)

# --- Recurso 2 de detalle, Problema 1: capacidad G1 ---
def Z_cap_g1(cap_g1):
    # Re-optimiza el problema 1 modificando únicamente la capacidad máxima instalada de la planta G1
    b = list(bounds_p1); b[0] = (0, cap_g1)
    r = linprog(c=c_p1, A_ub=A_ub_p1, b_ub=b_ub_p1, A_eq=A_eq_p1, b_eq=b_eq_p1, bounds=b, method='highs')
    return r.fun if r.success else np.inf

w_g1 = res_p1_scipy.upper.marginals[0]
rango_cap_g1 = encontrar_rango_precio_sombra("Capacidad G1 (P1)", Z_cap_g1, 260, w_g1, paso=1)

Emisiones (P1): precio sombra=-0.0260, rango de validez del RHS ≈ [210000.0, 300000.0]
Capacidad G1 (P1): precio sombra=-43.4000, rango de validez del RHS ≈ [222.0, 360.0]


In [19]:
# --- Tabla resumen del resto de recursos de Problema 1 (solo precio sombra puntual) ---
resumen_p1 = pd.DataFrame({
    "Recurso": ["Demanda", "Cuota renovable", "Take-or-pay G3", "Capacidad G2", "Capacidad G3", "Capacidad G4", "Capacidad G5"],
    "Precio sombra": [res_p1_dual.x[0], res_p1_dual.x[2], res_p1_dual.x[3],
                       res_p1_scipy.upper.marginals[1], res_p1_scipy.upper.marginals[2],
                       res_p1_scipy.upper.marginals[3], res_p1_scipy.upper.marginals[4]],
    "Unidades": ["miles USD/GWh"]*3 + ["miles USD/GWh"]*4
})
display(resumen_p1)

# --- Recurso 1 de detalle, Problema 2: Presupuesto ---
def Z_presupuesto_p2(presupuesto):
    r = linprog(c=c_p2, A_ub=A_ub_p2, b_ub=b_ub_p2, A_eq=[[1,1,1,1,1]], b_eq=[presupuesto],
                bounds=bounds_p2, method='highs')
    return -r.fun if r.success else np.inf

rango_presupuesto = encontrar_rango_precio_sombra("Presupuesto (P2)", Z_presupuesto_p2, 600, res_p2_dual.x[0], paso=1)

# --- Recurso 2 de detalle, Problema 2: Apetito de riesgo ---
def Z_riesgo_p2(tope_riesgo):
    r = linprog(c=c_p2, A_ub=[[9,5,2,6,1],[0,0,-1,0,-1],[0,0,0,-1,0]], b_ub=[tope_riesgo,-120,-50],
                A_eq=A_eq_p2, b_eq=b_eq_p2, bounds=bounds_p2, method='highs')
    return -r.fun if r.success else np.inf

rango_riesgo = encontrar_rango_precio_sombra("Apetito de riesgo (P2)", Z_riesgo_p2, 3300, res_p2_dual.x[1], paso=10)

# --- Tabla resumen del resto de recursos de Problema 2 ---
resumen_p2 = pd.DataFrame({
    "Recurso": ["Bajo riesgo (regulatorio)", "Meta PyME", "Límite L1", "Límite L2", "Límite L3", "Límite L4", "Límite L5"],
    "Precio sombra": [res_p2_dual.x[2], res_p2_dual.x[3],
                       res_p2_scipy.upper.marginals[0], res_p2_scipy.upper.marginals[1],
                       res_p2_scipy.upper.marginals[2], res_p2_scipy.upper.marginals[3],
                       res_p2_scipy.upper.marginals[4]],
    "Unidades": ["MUSD de rendimiento por MUSD/pto"]*2 + ["MUSD de rendimiento por millón USD"]*5
})
display(resumen_p2)

,Recurso,Precio sombra,Unidades
0,Demanda,55.4,miles USD/GWh
1,Cuota renovable,0.0,miles USD/GWh
2,Take-or-pay G3,0.0,miles USD/GWh
3,Capacidad G2,-47.4,miles USD/GWh
4,Capacidad G3,0.0,miles USD/GWh
5,Capacidad G4,0.0,miles USD/GWh
6,Capacidad G5,0.0,miles USD/GWh


Presupuesto (P2): precio sombra=0.0500, rango de validez del RHS ≈ [585.0, 710.0]
Apetito de riesgo (P2): precio sombra=0.0100, rango de validez del RHS ≈ [3180.0, 3330.0]


,Recurso,Precio sombra,Unidades
0,Bajo riesgo (regulatorio),0.000,MUSD de rendimiento por MUSD/pto
1,Meta PyME,0.000,MUSD de rendimiento por MUSD/pto
2,Límite L1,-0.025,MUSD de rendimiento por millón USD
3,Límite L2,-0.015,MUSD de rendimiento por millón USD
4,Límite L3,0.000,MUSD de rendimiento por millón USD
5,Límite L4,-0.020,MUSD de rendimiento por millón USD
6,Límite L5,0.000,MUSD de rendimiento por millón USD


### V.7 Pruebas de coherencia (mínimo 3, al menos una por problema, incluye análisis de unicidad)

In [20]:
# Prueba 1 (P1) — Monotonía: eliminar una restricción no puede empeorar el óptimo
r_sin_emisiones = linprog(c=c_p1, A_ub=A_ub_p1[1:], b_ub=b_ub_p1[1:],
                           A_eq=A_eq_p1, b_eq=b_eq_p1, bounds=bounds_p1, method='highs')
print(f"Prueba 1 (monotonía P1): Z*(con emisiones)={res_p1_scipy.fun:.1f} >= "
      f"Z*(sin emisiones)={r_sin_emisiones.fun:.1f} -> {res_p1_scipy.fun >= r_sin_emisiones.fun - 1e-6}")

# Prueba 2 (P1) — Caso extremo: tope de emisiones extremadamente laxo debe converger
# al óptimo sin la restricción de emisiones
r_laxo = linprog(c=c_p1, A_ub=[[0,0,400,900,700],[-1,-1,0,0,0],[0,0,-1,0,0]],
                  b_ub=[1e9, -312, -50], A_eq=A_eq_p1, b_eq=b_eq_p1, bounds=bounds_p1, method='highs')
print(f"Prueba 2 (caso extremo P1): Z*(tope~inf)={r_laxo.fun:.1f} == Z*(sin restr.)={r_sin_emisiones.fun:.1f} "
      f"-> {np.isclose(r_laxo.fun, r_sin_emisiones.fun)}")

# Prueba 3 (P2) — Verificación de unicidad: ¿hay variables no básicas con costo reducido nulo?
cr_p2 = np.where(res_p2_scipy.x < 1e-6, res_p2_scipy.lower.marginals, res_p2_scipy.upper.marginals)
no_basicas_alt = [(j+1, round(cr,6)) for j, (xj, cr) in enumerate(zip(res_p2_scipy.x, cr_p2))
                  if 1e-6 < xj < bounds_p2[j][1]-1e-6 and abs(cr) < 1e-6]
print(f"Prueba 3 (unicidad P2): variables básicas (0<x<límite) con costo reducido ~0: {no_basicas_alt}")
if no_basicas_alt:
    print("-> Existen soluciones óptimas alternativas; la recomendación de la Pregunta 9 debe matizarse.")
else:
    print("-> No se detectan variables no básicas con costo reducido nulo dentro de tolerancia: "
          "la evidencia apunta a una solución óptima única en los vértices, aunque esto no es una demostración formal.")

Prueba 1 (monotonía P1): Z*(con emisiones)=19870.0 >= Z*(sin emisiones)=18440.0 -> True
Prueba 2 (caso extremo P1): Z*(tope~inf)=18440.0 == Z*(sin restr.)=18440.0 -> True
Prueba 3 (unicidad P2): variables básicas (0<x<límite) con costo reducido ~0: [(3, np.float64(0.0)), (5, np.float64(0.0))]
-> Existen soluciones óptimas alternativas; la recomendación de la Pregunta 9 debe matizarse.


### Pregunta 9 — Decisiones de negocio, Problema 1

In [21]:
# --- 9(a): compra de derechos de emisión a 18 USD/tCO2 ---
precio_mercado_usd = 18
y_emisiones_usd = res_p1_dual.x[1] * 1000  # miles USD/tCO2 -> USD/tCO2
print(f"Precio sombra de emisiones: {abs(y_emisiones_usd):.2f} USD/tCO2 (ahorro por cada tCO2 adicional de cupo)")
print(f"Precio de mercado: {precio_mercado_usd} USD/tCO2")

if abs(y_emisiones_usd) > precio_mercado_usd:
    lo, hi = rango_emisiones  # ya calculado en V.6
    max_tCO2_comprar = hi - 245000
    ahorro_total_usd = abs(y_emisiones_usd) * max_tCO2_comprar
    print(f"SÍ conviene comprar. Máximo: {max_tCO2_comprar:.0f} tCO2, ahorro total: {ahorro_total_usd:.0f} USD")
    print("Más allá de ese punto el precio sombra cambia (otra restricción se activa) y ya no es "
          "correcto extrapolar el mismo ahorro marginal (error frecuente #10 del anexo).")
else:
    print("NO conviene comprar: el precio de mercado supera el ahorro marginal por tCO2.")

Precio sombra de emisiones: 26.00 USD/tCO2 (ahorro por cada tCO2 adicional de cupo)
Precio de mercado: 18 USD/tCO2
SÍ conviene comprar. Máximo: 55000 tCO2, ahorro total: 1430000 USD
Más allá de ese punto el precio sombra cambia (otra restricción se activa) y ya no es correcto extrapolar el mismo ahorro marginal (error frecuente #10 del anexo).


In [22]:
# --- 9(b): ampliación de capacidad — comparar G1 y G2 (únicas con precio sombra no nulo) ---
w_g1 = res_p1_scipy.upper.marginals[0]
w_g2 = res_p1_scipy.upper.marginals[1]
print(f"Precio sombra G1: {abs(w_g1):.2f} miles USD/GWh, rango válido hasta cap={rango_cap_g1[1]:.0f} GWh")

def Z_cap_g2(cap_g2):
    b = list(bounds_p1); b[1] = (0, cap_g2)
    r = linprog(c=c_p1, A_ub=A_ub_p1, b_ub=b_ub_p1, A_eq=A_eq_p1, b_eq=b_eq_p1, bounds=b, method='highs')
    return r.fun if r.success else np.inf

rango_cap_g2 = encontrar_rango_precio_sombra("Capacidad G2 (P1)", Z_cap_g2, 120, w_g2, paso=1)
print(f"Precio sombra G2: {abs(w_g2):.2f} miles USD/GWh, rango válido hasta cap={rango_cap_g2[1]:.0f} GWh")

if abs(w_g2) > abs(w_g1):
    print("\nConviene ampliar G2: mayor ahorro por GWh adicional, a pesar de que G1 y G2 no son "
          "las plantas de menor costo variable por separado (costo variable no es el criterio correcto).")
else:
    print("\nConviene ampliar G1.")
print("G3, G4, G5 tienen precio sombra 0 porque no están en su límite de capacidad en el óptimo "
      "(hay holgura), así que ampliarlas no cambia la solución aunque su costo variable sea distinto.")

Precio sombra G1: 43.40 miles USD/GWh, rango válido hasta cap=360 GWh
Capacidad G2 (P1): precio sombra=-47.4000, rango de validez del RHS ≈ [82.0, 220.0]
Precio sombra G2: 47.40 miles USD/GWh, rango válido hasta cap=220 GWh

Conviene ampliar G2: mayor ahorro por GWh adicional, a pesar de que G1 y G2 no son las plantas de menor costo variable por separado (costo variable no es el criterio correcto).
G3, G4, G5 tienen precio sombra 0 porque no están en su límite de capacidad en el óptimo (hay holgura), así que ampliarlas no cambia la solución aunque su costo variable sea distinto.


In [23]:
# --- 9(c): costo de la cláusula take-or-pay y de la cuota renovable ---
base = res_p1_scipy.fun

def Z_takeorpay(min_g3):
    r = linprog(c=c_p1, A_ub=A_ub_p1, b_ub=[245000,-312,-min_g3], A_eq=A_eq_p1, b_eq=b_eq_p1,
                bounds=bounds_p1, method='highs')
    return r.fun if r.success else np.inf

def Z_renovable(min_frac):
    min_gwh = min_frac*780
    r = linprog(c=c_p1, A_ub=A_ub_p1, b_ub=[245000,-min_gwh,-50], A_eq=A_eq_p1, b_eq=b_eq_p1,
                bounds=bounds_p1, method='highs')
    return r.fun if r.success else np.inf

print(f"Costo take-or-pay al umbral actual (50 GWh): {Z_takeorpay(50)-base:.1f} miles USD/semana")
print(f"Costo cuota renovable al umbral actual (40%): {Z_renovable(0.40)-base:.1f} miles USD/semana")

# Barrido para encontrar a partir de qué umbral cada cláusula empieza a costar
print("\nTake-or-pay: costo en función del mínimo de G3")
for g3min in [50,150,230,240,250,260]:
    z = Z_takeorpay(g3min)
    print(f"  min_G3={g3min}: costo={'inf (infactible)' if z==np.inf else round(z-base,1)}")

print("\nCuota renovable: costo en función del % mínimo")
for frac in [0.40,0.45,0.48,0.487,0.49]:
    z = Z_renovable(frac)
    print(f"  frac={frac}: costo={'inf (infactible)' if z==np.inf else round(z-base,1)}")

print("\nDiferencia cualitativa: el take-or-pay genera un costo que CRECE de forma lineal y suave en "
      "cuanto se endurece más allá de su punto actual (G3 ya opera en 230 GWh, así que cualquier mínimo "
      "por encima de 230 empieza a costar de inmediato). La cuota renovable, en cambio, tiene costo CERO "
      "en todo el rango hasta que G1+G2 topan su capacidad física combinada (380 GWh ≈ 48.7% de la demanda); "
      "más allá de ese punto el problema se vuelve INFACTIBLE de golpe, sin una transición gradual de costo.")

Costo take-or-pay al umbral actual (50 GWh): 0.0 miles USD/semana
Costo cuota renovable al umbral actual (40%): 0.0 miles USD/semana

Take-or-pay: costo en función del mínimo de G3
  min_G3=50: costo=0.0
  min_G3=150: costo=0.0
  min_G3=230: costo=0.0
  min_G3=240: costo=130.0
  min_G3=250: costo=260.0
  min_G3=260: costo=390.0

Cuota renovable: costo en función del % mínimo
  frac=0.4: costo=0.0
  frac=0.45: costo=0.0
  frac=0.48: costo=0.0
  frac=0.487: costo=0.0
  frac=0.49: costo=inf (infactible)

Diferencia cualitativa: el take-or-pay genera un costo que CRECE de forma lineal y suave en cuanto se endurece más allá de su punto actual (G3 ya opera en 230 GWh, así que cualquier mínimo por encima de 230 empieza a costar de inmediato). La cuota renovable, en cambio, tiene costo CERO en todo el rango hasta que G1+G2 topan su capacidad física combinada (380 GWh ≈ 48.7% de la demanda); más allá de ese punto el problema se vuelve INFACTIBLE de golpe, sin una transición gradual de costo.


### Pregunta 9 — Decisiones de negocio, Problema 2

In [24]:
# --- 9(a): dos ordenamientos de los límites de concentración ---
def Z_lim_p2(j, lim_j):
    b = list(bounds_p2); b[j] = (0, lim_j)
    r = linprog(c=c_p2, A_ub=A_ub_p2, b_ub=b_ub_p2, A_eq=A_eq_p2, b_eq=b_eq_p2, bounds=b, method='highs')
    return -r.fun if r.success else -np.inf

lims = [150,120,250,180,250]
nombres_l = ["L1","L2","L3","L4","L5"]
shadow_l = -res_p2_scipy.upper.marginals  # ganancia por millón adicional

def encontrar_rango_lim_p2(j, limbase, shadow, paso=1, max_iter=1000, tol=1e-4):
    Z0 = Z_lim_p2(j, limbase)
    d = 0
    while d < max_iter*paso:
        d += paso
        Z1 = Z_lim_p2(j, limbase+d)
        pred = Z0 + shadow*d
        if Z1 == -np.inf or abs(Z1-pred) > tol:
            break
    return limbase + d - paso

print("--- Ranking 1: por precio sombra (ganancia por millón adicional) ---")
ranking_shadow = sorted(zip(nombres_l, shadow_l), key=lambda t: -t[1])
for n, s in ranking_shadow:
    print(f"  {n}: {s:.4f} MUSD/millón")

print("\n--- Ranking 2: por ganancia total hasta el extremo del rango de validez ---")
ganancias_totales = {}
for j, (n, limbase, s) in enumerate(zip(nombres_l, lims, shadow_l)):
    if abs(s) < 1e-9:
        ganancias_totales[n] = 0.0
        continue
    bp = encontrar_rango_lim_p2(j, limbase, s, paso=1)
    ganancias_totales[n] = s * (bp - limbase)
    print(f"  {n}: límite {limbase} -> {bp:.0f}, ganancia total = {ganancias_totales[n]:.3f} MUSD")

ranking_total = sorted(ganancias_totales.items(), key=lambda t: -t[1])
print("\nRanking por ganancia total:", [n for n,_ in ranking_total])
print("Ranking por precio sombra:  ", [n for n,_ in ranking_shadow])
print("\nLos dos órdenes NO coinciden porque el precio sombra mide la ganancia MARGINAL (el primer millón "
      "adicional), mientras que la ganancia total depende también de CUÁNTO se puede ampliar ese límite "
      "antes de que otra restricción (el apetito de riesgo o el presupuesto) se vuelva la nueva activa. "
      "La pregunta del comité ('hasta dónde siga produciendo ganancia') pide la ganancia TOTAL, no la "
      "marginal, así que el criterio correcto es el Ranking 2.")

--- Ranking 1: por precio sombra (ganancia por millón adicional) ---
  L1: 0.0250 MUSD/millón
  L4: 0.0200 MUSD/millón
  L2: 0.0150 MUSD/millón
  L3: -0.0000 MUSD/millón
  L5: -0.0000 MUSD/millón

--- Ranking 2: por ganancia total hasta el extremo del rango de validez ---
  L1: límite 150 -> 165, ganancia total = 0.375 MUSD
  L2: límite 120 -> 150, ganancia total = 0.450 MUSD
  L4: límite 180 -> 204, ganancia total = 0.480 MUSD

Ranking por ganancia total: ['L4', 'L2', 'L1', 'L3', 'L5']
Ranking por precio sombra:   ['L1', 'L4', 'L2', 'L3', 'L5']

Los dos órdenes NO coinciden porque el precio sombra mide la ganancia MARGINAL (el primer millón adicional), mientras que la ganancia total depende también de CUÁNTO se puede ampliar ese límite antes de que otra restricción (el apetito de riesgo o el presupuesto) se vuelva la nueva activa. La pregunta del comité ('hasta dónde siga produciendo ganancia') pide la ganancia TOTAL, no la marginal, así que el criterio correcto es el Ranking 2.


In [25]:
# --- 9(b): tasa máxima rentable por 1 millón adicional de fondeo, dos escenarios ---
def Z_budget_p2(delta, flexible):
    b_eq = [600+delta]
    b_ub = [5.5*(600+delta), -120, -50] if flexible else [3300, -120, -50]
    r = linprog(c=c_p2, A_ub=A_ub_p2, b_ub=b_ub, A_eq=A_eq_p2, b_eq=b_eq, bounds=bounds_p2, method='highs')
    return -r.fun if r.success else -np.inf

z0 = Z_budget_p2(0, True)
z1_flex = Z_budget_p2(1, True)
z1_fixed = Z_budget_p2(1, False)
print(f"Escenario (i) apetito de riesgo flexible (crece proporcional): ganancia marginal = {z1_flex-z0:.4f} MUSD/millón")
print(f"Escenario (ii) apetito de riesgo fijo en 3300 pts:              ganancia marginal = {z1_fixed-z0:.4f} MUSD/millón")
print("\nEn ambos casos, esa ganancia marginal es la tasa MÁXIMA que el banco puede pagar por el fondeo "
      "adicional sin perder dinero. El escenario flexible rinde más porque relaja simultáneamente dos "
      "restricciones (presupuesto y riesgo), permitiendo desplazar la mezcla hacia líneas más rentables "
      "y más riesgosas (L1); en el escenario fijo, el riesgo ya no puede crecer, así que el millón "
      "adicional debe ir forzosamente a líneas de bajo riesgo, con menor rendimiento marginal.")

Escenario (i) apetito de riesgo flexible (crece proporcional): ganancia marginal = 0.1050 MUSD/millón
Escenario (ii) apetito de riesgo fijo en 3300 pts:              ganancia marginal = 0.0500 MUSD/millón

En ambos casos, esa ganancia marginal es la tasa MÁXIMA que el banco puede pagar por el fondeo adicional sin perder dinero. El escenario flexible rinde más porque relaja simultáneamente dos restricciones (presupuesto y riesgo), permitiendo desplazar la mezcla hacia líneas más rentables y más riesgosas (L1); en el escenario fijo, el riesgo ya no puede crecer, así que el millón adicional debe ir forzosamente a líneas de bajo riesgo, con menor rendimiento marginal.


In [26]:
# --- 9(c): eliminar la línea L5 ---
bounds_p2_noL5 = list(bounds_p2)
bounds_p2_noL5[4] = (0,0)
res_noL5 = linprog(c=c_p2, A_ub=A_ub_p2, b_ub=b_ub_p2, A_eq=A_eq_p2, b_eq=b_eq_p2, bounds=bounds_p2_noL5, method='highs')

base_p2 = -res_p2_scipy.fun
z_noL5 = -res_noL5.fun
print(f"Rendimiento con L5: {base_p2:.4f} MUSD")
print(f"Rendimiento sin L5: {z_noL5:.4f} MUSD")
print(f"Diferencia: {z_noL5-base_p2:.4f} MUSD")
print("Nueva asignación sin L5:", dict(zip(["L1","L2","L3","L4","L5"], res_noL5.x)))

print("\nEliminar L5 EMPEORA el resultado a pesar de tener el menor rendimiento (6%), porque L5 tiene "
      "también el menor índice de riesgo (1 pto/millón) y ayuda a cumplir la restricción de bajo riesgo "
      "(L3+L5>=120) sin consumir apetito de riesgo. Al quitarla, el presupuesto que antes iba a L5 debe "
      "reubicarse en líneas que consumen más riesgo por millón, obligando a sacrificar generación de "
      "rendimiento en otra parte del portafolio para no violar el tope de 3300 puntos. El argumento del "
      "analista ignora que el valor de una línea no depende solo de su rendimiento individual, sino del "
      "costo de oportunidad de los recursos que consume — que es exactamente lo que mide el problema dual.")

Rendimiento con L5: 72.1500 MUSD
Rendimiento sin L5: 72.0429 MUSD
Diferencia: -0.1071 MUSD
Nueva asignación sin L5: {'L1': np.float64(145.71428571428572), 'L2': np.float64(120.0), 'L3': np.float64(154.28571428571422), 'L4': np.float64(180.0), 'L5': np.float64(0.0)}

Eliminar L5 EMPEORA el resultado a pesar de tener el menor rendimiento (6%), porque L5 tiene también el menor índice de riesgo (1 pto/millón) y ayuda a cumplir la restricción de bajo riesgo (L3+L5>=120) sin consumir apetito de riesgo. Al quitarla, el presupuesto que antes iba a L5 debe reubicarse en líneas que consumen más riesgo por millón, obligando a sacrificar generación de rendimiento en otra parte del portafolio para no violar el tope de 3300 puntos. El argumento del analista ignora que el valor de una línea no depende solo de su rendimiento individual, sino del costo de oportunidad de los recursos que consume — que es exactamente lo que mide el problema dual.


### V.8 Registro y auditoría del uso de IA

**1. Prompts utilizados:** [Completar con tus prompts reales — resumen fiel, no hace falta pegarlos literales]

**2. Salida inicial recibida (versión 1 del notebook):** primal y dual resueltos correctamente para
ambos problemas (verificado numéricamente: Z*=19870 miles USD coincide con el ejemplo del propio
enunciado del taller), pero sin las verificaciones de V.1, V.4 (Problema 2), V.5, V.7, sin costos
reducidos (Pregunta 5), sin holgura complementaria (Pregunta 7), y sin ningún literal de la Pregunta 9.

**3. Error real detectado, con evidencia:**
Al forzar el take-or-pay a ser una restricción activa (min G3 ≥ 260) para poner a prueba la comparación
entre los multiplicadores del solver y el dual explícito (verificación V.3), se encontró que
`res.ineqlin.marginals` reporta **-13** para esa fila mientras que la variable dual "económica" del PL
dual explícito da **+13**. Es la misma cantidad con signo invertido, causada por la conversión de
`x3 ≥ 50` a `-x3 ≤ -50` que exige `linprog`. La salida inicial de la IA nunca hacía esta comparación
explícita ni advertía sobre el cambio de signo — es exactamente el error frecuente #5 que señala el
anexo del taller ("interpretar el signo de los duales de linprog al revés"). Se corrigió agregando la
sección de comparación (bloque de Pregunta 4/5) que documenta explícitamente esta convención de signos
para cada restricción convertida.

**4. Reflexión:** la verificación más efectiva para detectar este tipo de error no fue la comprobación
de dualidad fuerte (V.4) — que solo compara el valor objetivo total y por eso no detecta errores de
signo en multiplicadores individuales que se cancelan en la suma — sino la comparación fila por fila
entre `ineqlin.marginals` y el dual explícito (V.3), que sí expone discrepancias de signo aunque el
valor óptimo global sea idéntico.